# SentiStrength scores

Every review gets two scores from SentiStrength's word list:
- `senti_positive`: 1 (no positive feeling) up to 5 (very positive)
- `senti_negative`: -1 (no negative feeling) down to -5 (very negative)


##  Load the word lists

In [1]:
import re
import math
import pandas as pd

folder = "../data/SentiStrengthDataEnglishOctober2019/"

# Load the word list. Each line looks like "love<TAB>3"
word_scores = {}
for line in open(folder + "SentimentLookupTable.txt", encoding="latin-1"):
    word, score = line.split("\t")[:2]
    word_scores[word.lower()] = int(score)


# Idioms (phrases), e.g. "bite the dust<TAB>-3"
# stored as a list of words, e.g. (["bite", "the", "dust"], -3)
idioms = []
for line in open(folder + "IdiomLookupTable.txt", encoding="latin-1"):
    parts = line.strip().split("	")
    if len(parts) >= 2:
        idioms.append((parts[0].lower().split(), int(parts[1])))

# Words like "not", "don't", "never"
negations = open(folder + "NegatingWordList.txt", encoding="latin-1").read().lower().split()



##  Scoring functions

In [2]:
def get_word_score(word):
    # exact match, e.g. "love"
    if word in word_scores:
        return word_scores[word]
    # "*" means any ending, e.g. "lov*" also covers "loved" and "lovely"
    for w, score in word_scores.items():
        if w.endswith("*") and word.startswith(w[:-1]):
            return score
    return 0  # not in the list = no feeling


def score_sentence(sentence):
    words = re.findall(r"[a-z']+", sentence.lower())
    pos = 1   # 1 = no positive feeling
    neg = -1  # -1 = no negative feeling

    scores = [get_word_score(w) for w in words]

    # idioms replace the scores of their words: the first word gets the
    # idiom's score and the rest get 0, e.g. "bite the dust" -> -3, 0, 0
    for idiom_words, idiom_score in idioms:
        n = len(idiom_words)
        for i in range(len(words) - n + 1):
            if words[i:i + n] == idiom_words:
                scores[i] = idiom_score
                for j in range(i + 1, i + n):
                    scores[j] = 0

    for i in range(len(words)):
        score = scores[i]

        # "not good": flip a positive word to a weaker negative,
        # and "not bad" just cancels the negative word
        if i > 0 and words[i - 1] in negations:
            if score > 0:
                score = -math.ceil(score / 2)
            else:
                score = 0

        # keep the strongest positive and the strongest negative
        pos = max(pos, score)
        neg = min(neg, score)

    # "!" makes the stronger feeling one point stronger
    if "!" in sentence:
        if pos >= -neg:
            pos = pos + 1
        else:
            neg = neg - 1

    return pos, neg


def sentistrength(review):
    # cut the review into sentences (each one keeps its . ! or ? at the end)
    sentences = re.findall(r"[^.!?]+[.!?]*", review)

    # the review gets the strongest positive and negative of all its sentences
    pos, neg = 1, -1
    for sentence in sentences:
        p, n = score_sentence(sentence)
        pos = max(pos, p)
        neg = min(neg, n)

    # scores can't go beyond 5 / -5
    return min(pos, 5), max(neg, -5)


# try it on a few made-up reviews
for review in ["Great product!", "I do not like it", "Worst purchase ever", "Good shirt but terrible sizing",
               "This brand will bite the dust soon", "It is like silk"]:
    print(review, "->", sentistrength(review))

Great product! -> (4, -1)
I do not like it -> (1, -1)
Worst purchase ever -> (1, -3)
Good shirt but terrible sizing -> (3, -4)
This brand will bite the dust soon -> (1, -3)
It is like silk -> (1, -1)


## Score all the reviews

In [3]:
df = pd.read_excel("../data/Case3_Data1.xlsx")

# run every review through sentistrength() and split the result into two columns
scores = df["review_text"].apply(sentistrength)
df["senti_positive"] = [s[0] for s in scores]
df["senti_negative"] = [s[1] for s in scores]
df["net_score"] = df["senti_positive"] + df["senti_negative"]

result = df[["reviewer_id", "review_text", "star_rating", "senti_positive", "senti_negative", "net_score"]]
result.to_csv("../results/sentistrength_results.csv", index=False)
result

,reviewer_id,review_text,star_rating,senti_positive,senti_negative,net_score
0,2516,Great job problem solving,5,3,-2,1
1,3102,Constant Updating of Delivery information,4,1,-1,0
2,8694,On time delivery. No hassle ordering.,5,1,-1,0
3,9314,Easy to order.Great shipping.,5,3,-1,2
4,17105,I love it it fits perfectly and also my favori...,5,3,-1,2
...,...,...,...,...,...,...
195,273958,blow away mei,5,1,-1,0
196,274146,Sent 3 seperate emails to customer service reg...,1,1,-2,-1
197,275226,Would say AverageWasn't completely thrilled or...,3,4,-3,1
198,276779,"Have ordered from them twice, both times with ...",5,1,-1,0
